[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AImotion-Bavaria/kipro_wi/blob/main/lecture2/knapsack_ortools.ipynb)

# 0/1 Knapsack with OR-Tools

Choose items with **maximal total value** without exceeding the **capacity** of the knapsack.
Every item is either packed completely or not at all.

In [ ]:
!pip install -q ortools

## Data

```minizinc
set of int: ITEMS = 1..3;
array[ITEMS] of int: w;   % weights
int: capacity;
array[ITEMS] of int: v;   % values
```

Parameters are plain Python data. We use dictionaries to keep the indices `1..3` from MiniZinc.

In [ ]:
ITEMS = range(1, 4)

w = {1: 4, 2: 3, 3: 2}     # weights
v = {1: 10, 2: 7, 3: 5}    # values
capacity = 5

## Model

```minizinc
array[ITEMS] of var 0..1: x;
constraint sum(i in ITEMS) (x[i] * w[i]) <= capacity;
var int: f = sum(i in ITEMS) (x[i] * v[i]);
solve maximize f;
```

In [ ]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

# x[i] = 1 <=> item i is packed  (model.new_bool_var would work as well)
x = {i: model.new_int_var(0, 1, f"x[{i}]") for i in ITEMS}

# the packed items must fit into the knapsack
model.add(sum(x[i] * w[i] for i in ITEMS) <= capacity)

# objective: f is simply a linear expression, no extra variable needed
f = sum(x[i] * v[i] for i in ITEMS)
model.maximize(f)

## Solve

In [ ]:
solver = cp_model.CpSolver()
status = solver.solve(model)

if status == cp_model.OPTIMAL:
    packed = [i for i in ITEMS if solver.value(x[i])]
    print("x =", [solver.value(x[i]) for i in ITEMS])
    print("packed items:", packed)
    print("weight =", sum(w[i] for i in packed), "of", capacity)
    print("f =", solver.value(f))
else:
    print("no optimal solution found")

## Check: all possible packings

With 3 items there are only $2^3 = 8$ ways to pack, so we can list them all.
Note that packing the most valuable item 1 is **not** optimal:
it blocks the knapsack for the two smaller items, which together are worth more.

In [ ]:
from itertools import product

print(" x      weight  value  fits?")
for choice in product([0, 1], repeat=len(ITEMS)):
    weight = sum(c * w[i] for c, i in zip(choice, ITEMS))
    value = sum(c * v[i] for c, i in zip(choice, ITEMS))
    print(f"{list(choice)}  {weight:5}  {value:5}  {'yes' if weight <= capacity else 'no'}")

## Try it yourself

* Change `capacity` to 6 or 9 and run the notebook again (*Runtime → Run all*).
* Add a fourth item: extend `ITEMS` to `range(1, 5)` and add entries to `w` and `v`.